# DeBERTa-v3-small — Text Format Fine-Tuning (Experiment 1b)

**Task:** Binary classification of political statements as true (0) or false (1)  
**Model:** `microsoft/deberta-v3-small` — 86M parameters  
**Metric:** Macro F1 (forces recall on both classes despite 35/65 imbalance)

## Experiment context

This notebook implements **Experiment 1b** — the single most impactful design change in the transformer phase: prepending metadata as plain text tokens instead of encoding it through a parallel MLP network.

### The design choice: tokens vs MLP

| Approach | How metadata enters the model | Holdout Macro F1 |
|---|---|---|
| **Option A** — Text-only | Statement text alone | 0.6205 |
| **Option B** — Hybrid MLP | 4 float true-rate features via BatchNorm → Linear → GELU | 0.6133 (worse) |
| **Exp 1b** — Text format | Raw speaker / party / subject tokens prepended | **0.6392** |

The MLP branch failed because: (1) it introduces a second optimizer head, adding training instability; (2) float true-rate aggregations discard the specific names that DeBERTa's self-attention can cross-attend with the claim using pre-trained world knowledge about political figures and parties.

Prepending `"speaker: barack-obama | party: democrat | subject: foreign-policy | ..."` gives the model actual names — it can leverage what it learned about Obama or the Democratic party during pre-training, which no numeric encoding can replicate.

## Architecture

```
[formatted tokens] → DeBERTa-v3-small encoder → [CLS] (768-dim) → Dropout(0.3) → Linear(768 → 2)
```

No architecture change from the baseline — **only the input string changes**.

## Training strategy

- **No freeze** (`FREEZE_EPOCHS=0`): text-format input shifts the CLS representation enough that freezing the backbone for epoch 1 wastes capacity
- **LLRD** (Layer-wise LR Decay): head gets `LR=2e-5`; each encoder layer ×0.9 deeper — prevents well-trained lower layers from being overwritten
- **Epochs = 3**: confirmed sweet spot; ep4–5 overfit on 8k samples
- **Class weights** `[1.42, 0.77]`: compensates for 35/65 imbalance at the loss level

In [ ]:
from datetime import datetime
from pathlib import Path
import sys
from time import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from torch.amp import autocast
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

## Environment & Configuration

The script auto-detects Kaggle vs local execution. On Kaggle the dataset lives at `/kaggle/input/truth-classifier-nlp`; locally it walks upward from `cwd()` until it finds the project root (the directory containing `data/train.csv`).

**Key hyperparameters (Experiment 1b):**

| Parameter | Value | Why |
|---|---|---|
| `MODEL_NAME` | `deberta-v3-small` | Best English NLI/classification encoder; SentencePiece handles subwords better than WordPiece |
| `MAX_LENGTH` | 128 | Prefix adds ~12 tokens; p99 statement length ≈ 41; 128 leaves comfortable headroom |
| `BATCH_SIZE` | 16 | Fits 12 GB VRAM in FP32; larger batches risk noisy gradients on 8k samples |
| `EPOCHS` | 3 | Sweep confirmed ep3 is best; ep4–5 overfit |
| `FREEZE_EPOCHS` | **0** | No freeze — text-format representation shifts CLS enough that freezing ep1 wastes capacity |
| `CLS_DROPOUT` | 0.3 | Regularises the linear head; higher than HuggingFace default (0.1) to match small dataset |
| `LR` | 2e-5 | Standard fine-tuning range for DeBERTa; higher rates caused instability in prior runs |
| `LLRD_FACTOR` | 0.9 | Each encoder layer deeper gets `lr × 0.9^depth`; embeddings barely move from pre-trained state |
| `CLASS_WEIGHTS` | [1.42, 0.77] | Inverse of class frequencies: 35.25% true (0), 64.75% false (1) |

In [ ]:
IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_DIR   = Path("/kaggle/input/truth-classifier-nlp")
    OUTPUT_DIR = Path("/kaggle/working/models/transformer_textformat")
else:
    def _find_root(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "data" / "train.csv").exists():
                return p
        raise FileNotFoundError("Cannot find project root")
    _root      = _find_root(Path.cwd())
    DATA_DIR   = _root / "data"
    OUTPUT_DIR = _root / "models" / "transformer_textformat"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME    = "microsoft/deberta-v3-small"
MAX_LENGTH    = 128
BATCH_SIZE    = 16
EPOCHS        = 3
FREEZE_EPOCHS = 0
CLS_DROPOUT   = 0.3
LR            = 2e-5
LLRD_FACTOR   = 0.9
WARMUP_RATIO  = 0.1
WEIGHT_DECAY  = 0.01

CLASS_WEIGHTS = [1.42, 0.77]
THRESHOLD     = 0.5
SEED          = 42

enable_threshold_tuning = True
create_kaggle_csv       = True
model_slug              = "deberta-v3-small-textformat"

NUM_WORKERS = 0 if sys.platform == "win32" else 2

torch.manual_seed(SEED)
np.random.seed(SEED)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = False  # DeBERTa-v3 is unstable in BF16; FP32 throughout

print(f"Device : {device}")
if device.type == "cuda":
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  AMP  : {USE_AMP}")

_script_start = time()
def _now() -> str:
    return datetime.now().strftime("%H:%M:%S")

## Input Formatter

The `format_input` function converts structured columns into the single string the tokenizer sees. Key decisions:

- **Primary subject only**: the subject column is comma-separated multi-label (e.g. `"health-care,taxes"`); taking only the first topic avoids length blowup and keeps the topic signal focused
- **Lowercase all fields**: DeBERTa's SentencePiece vocabulary handles mixed case, but speaker names arrive as `"Barack Obama"` or `"barack-obama"` depending on source — normalising prevents the same entity appearing under multiple surface forms
- **NaN fallback to `"unknown"`**: ~3% of rows have missing speaker or party; `"unknown"` is a real vocabulary token, unlike empty strings or `"nan"`
- **Pipe `|` separator**: visually distinct from commas and hyphens in speaker names; avoids the tokenizer merging adjacent fields into unintended subwords

Example output:
```
speaker: barack-obama | party: democrat | subject: foreign-policy | Says the Affordable Care Act will add to the deficit.
```

In [ ]:
def format_input(speaker: str, party: str, subject: str, statement: str) -> str:
    """Prepend speaker/party/primary-subject as tokens before the statement."""
    def _clean(val) -> str:
        if pd.isna(val) or str(val).strip() == "":
            return "unknown"
        return str(val).strip().lower()

    primary_subject = _clean(subject).split(",")[0].strip()
    return (
        f"speaker: {_clean(speaker)} | "
        f"party: {_clean(party)} | "
        f"subject: {primary_subject} | "
        f"{statement}"
    )

## Data Loading & Format Inspection

We print a live example and the token-length distribution to verify that `MAX_LENGTH=128` comfortably covers the formatted inputs. The prefix (`"speaker: X | party: Y | subject: Z | "`) typically adds 10–14 tokens, so the effective statement budget is ~115 tokens — well above p99 of statement length (~41–55 tokens).

Token-length statistics use whitespace-split word counts as a fast approximation (actual BPE token counts are slightly higher but proportional).

In [ ]:
print(f"[{_now()}] Loading data...")
df = pd.read_csv(DATA_DIR / "train.csv")
print(f"  Rows: {len(df):,}  |  Labels: {df['label'].value_counts().to_dict()}")

example_row  = df.iloc[0]
example_text = format_input(
    example_row["speaker"], example_row["party_affiliation"],
    example_row["subject"],  example_row["statement"],
)
print(f"\nFormat  : 'speaker: {{speaker}} | party: {{party}} | subject: {{subject}} | {{statement}}'")
print(f"Example : {example_text[:140]}{'...' if len(example_text) > 140 else ''}")

tok_raw    = df["statement"].str.split().str.len()
tok_prefix = df.apply(
    lambda r: len(format_input(
        r["speaker"], r["party_affiliation"], r["subject"], r["statement"]
    ).split()),
    axis=1,
)
print(f"\n  Token len (statement only) : min={tok_raw.min()}  median={tok_raw.median():.0f}  p99={tok_raw.quantile(0.99):.0f}  max={tok_raw.max()}")
print(f"  Token len (with prefix)    : min={tok_prefix.min()}  median={tok_prefix.median():.0f}  p99={tok_prefix.quantile(0.99):.0f}  max={tok_prefix.max()}")
print(f"  MAX_LENGTH={MAX_LENGTH} covers {(tok_prefix <= MAX_LENGTH).mean()*100:.1f}% of samples")

## Train / Validation / Holdout Split

The 80/20 holdout split uses `random_state=42` and stratification — **identical to every other model in this project**. This is critical: without the same holdout all models can be fairly compared on the same rows.

```
all data (8,950)
├── trainval 80% (~7,160)  →  train 90% (~6,444)  |  val 10% (~716)
└── holdout  20% (~1,790)   ← never seen during training or threshold tuning
```

The validation split is used for checkpoint selection (best epoch) and threshold tuning. The holdout is touched exactly once — at final evaluation.

In [ ]:
all_texts  = df.apply(
    lambda r: format_input(
        r["speaker"], r["party_affiliation"], r["subject"], r["statement"]
    ),
    axis=1,
).tolist()
all_labels = df["label"].tolist()
all_idx    = np.arange(len(df))

idx_tv, idx_ho = train_test_split(
    all_idx, test_size=0.2, random_state=SEED, stratify=all_labels
)
idx_tr, idx_val = train_test_split(
    idx_tv, test_size=0.1, random_state=SEED,
    stratify=[all_labels[i] for i in idx_tv]
)

X_tr,  y_tr  = [all_texts[i] for i in idx_tr],  [all_labels[i] for i in idx_tr]
X_val, y_val = [all_texts[i] for i in idx_val], [all_labels[i] for i in idx_val]
X_ho,  y_ho  = [all_texts[i] for i in idx_ho],  [all_labels[i] for i in idx_ho]

print(f"Split — Train: {len(X_tr):,}   Val: {len(X_val):,}   Holdout: {len(X_ho):,}")

## Tokenizer & Dataset

`AutoTokenizer.from_pretrained` loads DeBERTa-v3's SentencePiece tokenizer. Key settings:

- **`truncation=True`**: inputs longer than `MAX_LENGTH` are right-truncated (rare — p99 fits easily)
- **`padding="max_length"`**: shorter sequences are padded so all tensors in a batch have identical shape — required for `DataLoader` collation without a custom collate function
- **`return_tensors="pt"`**: tokenises the entire split at once and returns PyTorch tensors; faster than lazy per-sample tokenisation in `__getitem__`

Tokenising the full split at construction time shifts I/O cost out of the training loop. The trade-off is up-front RAM usage — acceptable for 8k samples.

`val_loader` and `holdout_loader` use `BATCH_SIZE*2`: no gradients are stored during evaluation, halving the VRAM footprint and allowing a larger batch.

In [ ]:
print(f"[{_now()}] Tokenizing...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class StatementDataset(Dataset):
    def __init__(self, texts: list[str], labels: list[int]):
        self.enc    = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=MAX_LENGTH, return_tensors="pt"
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        return {k: v[idx] for k, v in self.enc.items()}, self.labels[idx]


_t0 = time()
train_ds   = StatementDataset(X_tr,  y_tr)
val_ds     = StatementDataset(X_val, y_val)
holdout_ds = StatementDataset(X_ho,  y_ho)
print(f"  Tokenized in {time()-_t0:.1f}s")

_pin = USE_AMP
train_loader   = DataLoader(train_ds,   batch_size=BATCH_SIZE,   shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=_pin)
val_loader     = DataLoader(val_ds,     batch_size=BATCH_SIZE*2, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=_pin)
holdout_loader = DataLoader(holdout_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=_pin)

print(f"  Train batches: {len(train_loader)}   Val batches: {len(val_loader)}   Holdout batches: {len(holdout_loader)}")

## Layer-wise Learning Rate Decay (LLRD)

LLRD assigns a different learning rate to each encoder layer, decreasing exponentially toward the input:

```
Classification head   → LR = 2e-5          (highest — learns from scratch)
Encoder layer 11      → LR = 2e-5 × 0.9¹ = 1.80e-5
Encoder layer 10      → LR = 2e-5 × 0.9² = 1.62e-5
  ...
Encoder layer 0       → LR = 2e-5 × 0.9¹¹ = 0.64e-5
Embeddings            → LR = 2e-5 × 0.9¹² = 0.58e-5  (lowest — stay near pre-trained)
```

**Why it helps:** A single global LR risks overwriting the rich semantic representations in lower layers that took pre-training to build. LLRD lets the classification head adapt aggressively while keeping the foundation layers close to their pre-trained weights.

**Bias and LayerNorm parameters** are excluded from weight decay (standard practice) — they are placed in separate param groups with `weight_decay=0`.

In [ ]:
def _build_llrd_param_groups(model, base_lr: float, llrd_factor: float,
                              weight_decay: float) -> list:
    no_decay   = {"bias", "LayerNorm.weight", "layer_norm.weight"}
    num_layers = len(model.deberta.encoder.layer)
    param_dict = dict(model.named_parameters())
    assigned   = set()
    groups     = []

    def _add(names: list[str], lr: float) -> None:
        wd = [param_dict[n] for n in names if not any(nd in n for nd in no_decay)]
        nd = [param_dict[n] for n in names if     any(nd in n for nd in no_decay)]
        if wd: groups.append({"params": wd, "lr": lr, "weight_decay": weight_decay})
        if nd: groups.append({"params": nd, "lr": lr, "weight_decay": 0.0})
        assigned.update(names)

    # Classification head + pooler (highest LR)
    head = [n for n in param_dict
            if "deberta.encoder.layer." not in n and "deberta.embeddings." not in n]
    _add(head, base_lr)

    # Encoder layers — higher index = shallower = higher LR
    for layer_idx in range(num_layers - 1, -1, -1):
        depth = num_layers - layer_idx
        lr    = base_lr * (llrd_factor ** depth)
        _add([n for n in param_dict if f"deberta.encoder.layer.{layer_idx}." in n], lr)

    # Embeddings (lowest LR)
    embed_lr = base_lr * (llrd_factor ** (num_layers + 1))
    _add([n for n in param_dict if "deberta.embeddings." in n and n not in assigned],
         embed_lr)

    return groups

## Freeze / Unfreeze Helpers

With `FREEZE_EPOCHS=0` (Experiment 1b), these helpers are **not called** — the LLRD optimizer runs unfrozen from epoch 1. They are kept here because:

1. Experiment 1 (`FREEZE_EPOCHS=1`) was the ablation that showed freezing hurts with text-format input, making the comparison informative
2. The same helpers are reused in `transformer_kfold.py`

**The freeze principle:** freeze the encoder in early epochs so only the randomly-initialised classification head trains first. Once the head stabilises, unfreeze and fine-tune everything. This prevents the noisy initial head gradients from corrupting the pre-trained encoder representations. The text-format experiment found this unnecessary — the shifted CLS representation already regularises training sufficiently.

In [ ]:
def _freeze_backbone(model) -> None:
    for name, param in model.named_parameters():
        if "classifier" not in name and "pooler" not in name:
            param.requires_grad_(False)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Backbone frozen — frozen={frozen:,}  trainable={trainable:,}")


def _unfreeze_backbone(model) -> None:
    for param in model.parameters():
        param.requires_grad_(True)
    print(f"  Backbone unfrozen — trainable={sum(p.numel() for p in model.parameters()):,}")

## Model Initialisation

`AutoModelForSequenceClassification` wraps the DeBERTa encoder with a 2-class linear head. Config settings:

- **`num_labels=2`**: binary classification — the head is `Linear(768, 2)` producing two logits; softmax gives class probabilities
- **`cls_dropout=0.3`**: applied to the `[CLS]` representation before the linear head; higher than the HuggingFace default (0.1) to regularise on 8k samples
- **`torch_dtype=torch.float32`**: DeBERTa-v3's disentangled attention mechanism is numerically unstable in BF16; FP32 is required throughout

**Class-weighted loss:** `CrossEntropyLoss(weight=[1.42, 0.77])` makes the model pay 1.84× more attention to errors on the minority class (true=0), counteracting the 65/35 imbalance without modifying the data or sampling strategy.

In [ ]:
print(f"[{_now()}] Loading model: {MODEL_NAME}")
_cfg = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
_cfg.cls_dropout = CLS_DROPOUT
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=_cfg, torch_dtype=torch.float32,
)
model.to(device)
print(f"  Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

loss_weights = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32).to(device)
criterion    = nn.CrossEntropyLoss(weight=loss_weights)

if FREEZE_EPOCHS > 0:
    _freeze_backbone(model)
    p1_steps  = len(train_loader) * FREEZE_EPOCHS
    p1_warmup = int(p1_steps * WARMUP_RATIO)
    p1_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(p1_params, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(optimizer, p1_warmup, p1_steps)
    p2_steps  = len(train_loader) * (EPOCHS - FREEZE_EPOCHS)
    p2_warmup = int(p2_steps * WARMUP_RATIO)
    print(f"  Phase 1 optimizer (frozen) — {len(p1_params)} param tensors  lr={LR:.1e}")
else:
    p2_steps  = len(train_loader) * EPOCHS
    p2_warmup = int(p2_steps * WARMUP_RATIO)
    param_groups = _build_llrd_param_groups(model, LR, LLRD_FACTOR, WEIGHT_DECAY)
    _lr_min = min(g["lr"] for g in param_groups)
    _lr_max = max(g["lr"] for g in param_groups)
    optimizer    = AdamW(param_groups)
    scheduler    = get_linear_schedule_with_warmup(optimizer, p2_warmup, p2_steps)
    print(f"  LLRD optimizer (no freeze) — {len(param_groups)} groups  LR range: [{_lr_min:.2e}, {_lr_max:.2e}]")

print(f"  Loss weights: {loss_weights.cpu().numpy()}")

## Training Helpers

**`train_epoch`:**
- `clip_grad_norm_(1.0)`: gradient clipping prevents exploding gradients, especially important in early epochs when the head is randomly initialised
- `scheduler.step()` is called after every batch (not every epoch) — the linear warmup schedule increases LR from 0 to peak over the first 10% of total steps, then decays linearly to 0
- `USE_AMP=False`: DeBERTa-v3 produces NaN logits in BF16 due to its disentangled attention; FP32 is safer and the VRAM penalty is acceptable at batch=16

**`evaluate`:**
- `@torch.no_grad()`: disables gradient tracking; ~2× faster and ~50% lower VRAM during inference
- `logits.float().cpu()`: cast to FP32 before accumulation to avoid overflow in `softmax`
- Returns raw **probabilities for class 1** (false) — not hard predictions — so threshold tuning can be applied post-hoc

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion) -> float:
    model.train()
    total_loss = 0.0
    for batch_idx, (inputs, labs) in enumerate(loader):
        inputs = {k: v.to(device) for k, v in inputs.items()}
        labs   = labs.to(device)
        optimizer.zero_grad()
        with autocast("cuda", dtype=torch.bfloat16, enabled=USE_AMP):
            logits = model(**inputs).logits
            loss   = criterion(logits, labs)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if batch_idx == 0:
            print(f"    Batch 0 — logits dtype={logits.dtype}  loss={loss.item():.4f}")
        if (batch_idx + 1) % 50 == 0:
            print(f"    Batch {batch_idx+1}/{len(loader)}  avg_loss={total_loss/(batch_idx+1):.4f}")
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    all_logits, all_labels, total_loss = [], [], 0.0
    for inputs, labs in loader:
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with autocast("cuda", dtype=torch.bfloat16, enabled=USE_AMP):
            logits = model(**inputs).logits
            loss   = criterion(logits, labs.to(device))
        total_loss += loss.item()
        all_logits.append(logits.float().cpu())
        all_labels.append(labs)
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels).numpy()
    proba  = torch.softmax(logits, dim=-1)[:, 1].numpy()
    return total_loss / len(loader), proba, labels

## Training Loop

Each epoch:
1. Runs `train_epoch` — full pass over training data with gradient updates
2. Runs `evaluate` on the validation set — no gradient computation
3. Computes `val_macro_f1` at threshold=0.5 for checkpoint selection
4. Saves model state dict if val macro_f1 improved

**Checkpoint selection by val macro_f1** (not val loss): loss can continue decreasing as the model learns well-calibrated probabilities for the majority class only — val macro_f1 directly measures what we care about, forcing recall on both classes.

**Phase 2 transition:** if `FREEZE_EPOCHS > 0`, the LLRD optimizer is rebuilt at the right epoch and the backbone unfreezes. With `FREEZE_EPOCHS=0` (Experiment 1b) this branch is never entered.

In [ ]:
print(f"[{_now()}] Starting training")
print(f"  Model dtype   : {next(model.parameters()).dtype}")
print(f"  Loss weights  : {loss_weights.cpu().numpy()}")
print(f"  Train batches : {len(train_loader)}  Val batches: {len(val_loader)}")

best_val_f1 = -1.0
best_ckpt   = OUTPUT_DIR / f"{model_slug}-best.pt"
epoch_log   = []

for epoch in range(1, EPOCHS + 1):
    _t = time()

    if FREEZE_EPOCHS > 0 and epoch == FREEZE_EPOCHS + 1:
        print(f"\n  [Phase 2] Unfreezing backbone + rebuilding optimizer with LLRD")
        _unfreeze_backbone(model)
        param_groups = _build_llrd_param_groups(model, LR, LLRD_FACTOR, WEIGHT_DECAY)
        optimizer    = AdamW(param_groups)
        _lr_min = min(g["lr"] for g in param_groups)
        _lr_max = max(g["lr"] for g in param_groups)
        scheduler    = get_linear_schedule_with_warmup(optimizer, p2_warmup, p2_steps)
        print(f"  LLRD groups: {len(param_groups)}  LR range: [{_lr_min:.2e}, {_lr_max:.2e}]")

    print(f"\n  --- Epoch {epoch}/{EPOCHS} ---  [{_now()}]")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion)
    print(f"  Train loss: {train_loss:.4f}  — evaluating on val set")

    val_loss, val_proba, val_labels = evaluate(model, val_loader)
    print(f"  Val proba range: [{val_proba.min():.4f}, {val_proba.max():.4f}]  NaNs: {np.isnan(val_proba).sum()}")

    val_pred = (val_proba >= 0.5).astype(int)
    val_f1   = f1_score(val_labels, val_pred, average="macro", zero_division=0)
    val_auc  = roc_auc_score(val_labels, val_proba)

    elapsed = time() - _t
    print(
        f"  Epoch {epoch}/{EPOCHS}  "
        f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
        f"val_macro_f1={val_f1:.4f}  val_roc_auc={val_auc:.4f}  "
        f"({elapsed:.1f}s)"
    )
    epoch_log.append({"epoch": epoch, "train_loss": train_loss,
                      "val_loss": val_loss, "val_f1": val_f1, "val_auc": val_auc})

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_ckpt)
        print(f"    New best val macro_f1={best_val_f1:.4f} — checkpoint saved")

print(f"\nTraining complete. Best val macro_f1 = {best_val_f1:.4f}")
display(pd.DataFrame(epoch_log).set_index("epoch").round(4))

## Loading Best Checkpoint

After all epochs, reload the best-performing checkpoint (by val macro_f1) before threshold tuning and holdout evaluation. This is important because the final epoch may have overfit — in experiments with 5 epochs, epochs 4–5 consistently degraded below epoch 3.

In Experiment 1b the best epoch was consistently epoch 3, but the checkpoint mechanism makes this robust to future hyperparameter changes.

In [ ]:
print(f"[{_now()}] Loading best checkpoint from {best_ckpt}")
model.load_state_dict(torch.load(best_ckpt, map_location=device))
print(f"  Reloaded checkpoint (best val macro_f1={best_val_f1:.4f})")

# Re-evaluate on val set to confirm proba consistency after reload
_, val_proba, val_labels = evaluate(model, val_loader)
print(f"  Val proba range after reload: [{val_proba.min():.4f}, {val_proba.max():.4f}]")

## Threshold Tuning

The default 0.5 threshold is not optimal for imbalanced data. With 65% majority class (false=1), the model's raw probabilities are skewed toward predicting 1. Sweeping thresholds from 0.20 to 0.75 and selecting the one that maximises **val macro_f1** recovers recall on the minority class (true=0).

**Why val set, not holdout:** using the holdout for threshold selection would constitute data leakage — the holdout must remain unseen until final evaluation. The same threshold is then applied identically to the holdout and the Kaggle test set.

Intuitively: a lower threshold means the model only needs to be slightly confident to predict "false" (1). Moving the threshold down from 0.5 makes the model less aggressive at predicting the majority class, forcing it to predict "true" (0) more often.

In [ ]:
if enable_threshold_tuning:
    print(f"[{_now()}] Threshold sweep on val set")
    grid   = np.arange(0.20, 0.76, 0.01)
    scores = {
        round(float(t), 2): f1_score(
            val_labels, (val_proba >= t).astype(int),
            average="macro", zero_division=0
        )
        for t in grid
    }
    best_t = max(scores, key=scores.get)

    print(f"  {'threshold':>10}   macro_f1")
    for t, s in scores.items():
        print(f"  {t:>10.2f}   {s:.4f}{'  <--' if t == best_t else ''}")
    print(f"\n  Best threshold: {best_t:.2f}  (val macro_f1={scores[best_t]:.4f})")
    THRESHOLD = best_t

    fig_thr, ax_thr = plt.subplots(figsize=(10, 4))
    ax_thr.plot(list(scores.keys()), list(scores.values()), "b-o", markersize=3, label="Val Macro F1")
    ax_thr.axvline(best_t, color="red", linestyle="--", label=f"Best = {best_t:.2f}  (F1={scores[best_t]:.4f})")
    ax_thr.axvline(0.5, color="gray", linestyle=":", alpha=0.8, label="Default 0.5")
    ax_thr.set_xlabel("Classification Threshold")
    ax_thr.set_ylabel("Val Macro F1")
    ax_thr.set_title("Threshold Sweep — Val Macro F1 vs Threshold")
    ax_thr.legend()
    plt.tight_layout()
    plt.show()

print(f"  Using THRESHOLD = {THRESHOLD:.2f}")

## Holdout Evaluation

The holdout (20% of the dataset, never seen during training or threshold tuning) gives the final unbiased estimate of model performance.

Key metrics to watch:
- **Macro F1**: primary competition metric — unweighted average of F1 for class 0 (true) and class 1 (false)
- **ROC-AUC**: threshold-independent measure of the model's discrimination power
- **Class-wise recall**: if class 0 recall is very low (<0.50), the model is effectively ignoring the minority class despite class weights and threshold adjustment
- **MCC** (Matthews Correlation Coefficient): a single balanced metric robust to class imbalance; values near 0 indicate chance-level performance

In [ ]:
print(f"[{_now()}] Holdout evaluation  (threshold={THRESHOLD:.2f})")

_, ho_proba, ho_labels = evaluate(model, holdout_loader)
ho_pred = (ho_proba >= THRESHOLD).astype(int)

holdout_metrics = {
    "roc_auc":      roc_auc_score(ho_labels, ho_proba),
    "pr_auc":       average_precision_score(ho_labels, ho_proba),
    "macro_f1":     f1_score(ho_labels, ho_pred, average="macro", zero_division=0),
    "f1":           f1_score(ho_labels, ho_pred, zero_division=0),
    "precision":    precision_score(ho_labels, ho_pred, zero_division=0),
    "recall":       recall_score(ho_labels, ho_pred, zero_division=0),
    "accuracy":     accuracy_score(ho_labels, ho_pred),
    "mcc":          matthews_corrcoef(ho_labels, ho_pred),
    "balanced_acc": balanced_accuracy_score(ho_labels, ho_pred),
}
cm = confusion_matrix(ho_labels, ho_pred)

print("\nHoldout results:")
for name, val in holdout_metrics.items():
    print(f"  {name:15s}: {val:.4f}")
print(f"\n{classification_report(ho_labels, ho_pred, target_names=['True (0)', 'False (1)'])}")

## Evaluation Plots

Three standard diagnostic plots:

- **ROC Curve**: true positive rate vs false positive rate across all thresholds; area under curve (ROC-AUC) is threshold-independent  
- **Precision-Recall Curve**: more informative than ROC for imbalanced datasets — a random classifier's PR-AUC equals the positive class prevalence (~0.65)
- **Confusion Matrix**: absolute counts of TN / FP / FN / TP at the selected threshold; shows directly whether class 0 (true) is being predicted or ignored

In [ ]:
fpr, tpr, _      = roc_curve(ho_labels, ho_proba)
prec_c, rec_c, _ = precision_recall_curve(ho_labels, ho_proba)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(fpr, tpr, label=f"ROC-AUC = {holdout_metrics['roc_auc']:.4f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.6)
axes[0].set(title=f"ROC Curve — {model_slug} (holdout)",
            xlabel="False Positive Rate", ylabel="True Positive Rate")
axes[0].legend()

axes[1].plot(rec_c, prec_c, label=f"PR-AUC = {holdout_metrics['pr_auc']:.4f}")
axes[1].set(title="Precision-Recall Curve (holdout)",
            xlabel="Recall", ylabel="Precision")
axes[1].legend()

im = axes[2].imshow(cm, interpolation="nearest", cmap="Blues")
axes[2].set_title("Confusion Matrix (holdout)")
axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(["True (0)", "False (1)"])
axes[2].set_yticks([0, 1]); axes[2].set_yticklabels(["True (0)", "False (1)"])
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
fig.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

## Saving Artifacts

`model.save_pretrained` and `tokenizer.save_pretrained` write the full HuggingFace model directory (config, weights, tokenizer files). The threshold is saved separately with `joblib` so inference scripts can load it independently without loading the full model.

These artifacts are sufficient to reproduce inference:
```python
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR / "deberta-v3-small-textformat-tokenizer")
model     = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR / "deberta-v3-small-textformat-model")
threshold = joblib.load(OUTPUT_DIR / "deberta-v3-small-textformat-threshold.joblib")
```

In [ ]:
print(f"[{_now()}] Saving artifacts to {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR / f"{model_slug}-model")
tokenizer.save_pretrained(OUTPUT_DIR / f"{model_slug}-tokenizer")
joblib.dump(THRESHOLD, OUTPUT_DIR / f"{model_slug}-threshold.joblib")
print(f"  Model     : {OUTPUT_DIR / f'{model_slug}-model'}")
print(f"  Tokenizer : {OUTPUT_DIR / f'{model_slug}-tokenizer'}")
print(f"  Threshold : {OUTPUT_DIR / f'{model_slug}-threshold.joblib'}  (value={THRESHOLD:.2f})")

## Kaggle Submission

Inference on the unlabelled test set applies the same `format_input` function and `MAX_LENGTH=128` settings used during training. The batched inference loop uses `BATCH_SIZE*2` — no gradients needed, so we can afford a larger batch and reduce inference time.

The probability threshold applied here is exactly the one tuned on the validation set, ensuring consistency between offline evaluation and submission.

In [ ]:
if create_kaggle_csv:
    print(f"[{_now()}] Creating Kaggle submission")
    df_test    = pd.read_csv(DATA_DIR / "test_nolabel.csv")
    test_texts = df_test.apply(
        lambda r: format_input(
            r["speaker"], r["party_affiliation"], r["subject"], r["statement"]
        ),
        axis=1,
    ).tolist()

    test_enc = tokenizer(
        test_texts, truncation=True, padding="max_length",
        max_length=MAX_LENGTH, return_tensors="pt",
    )

    model.eval()
    all_proba = []
    _bsz = BATCH_SIZE * 2
    with torch.no_grad():
        for i in range(0, len(test_texts), _bsz):
            batch = {k: v[i:i + _bsz].to(device) for k, v in test_enc.items()}
            with autocast("cuda", dtype=torch.bfloat16, enabled=USE_AMP):
                logits = model(**batch).logits
            all_proba.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())

    test_proba = np.concatenate(all_proba)
    test_pred  = (test_proba >= THRESHOLD).astype(int)

    sub_dir = Path("/kaggle/working") if IS_KAGGLE else (_root / "submissions")
    sub_dir.mkdir(exist_ok=True)
    sub_path = sub_dir / f"submission-{model_slug}-{datetime.now().strftime('%Y%m%d-%H%M')}.csv"
    pd.DataFrame({"id": df_test["id"], "label": test_pred}).to_csv(sub_path, index=False)
    print(f"  Submission saved: {sub_path}  ({len(test_pred):,} rows)")
    print(f"  Predicted class distribution: {pd.Series(test_pred).value_counts().to_dict()}")

print(f"\n[DONE] Total time: {time()-_script_start:.1f}s  [{_now()}]")

## Summary

### Results

| Metric | Value |
|---|---|
| Holdout Macro F1 | **0.6392** |
| Holdout ROC-AUC | **0.6885** |
| vs Option A (text-only baseline) | +0.0187 F1 &nbsp; +0.0185 AUC |
| vs Option B (hybrid MLP) | +0.0259 F1 &nbsp; (MLP was worse than text-only) |

### Key findings

1. **Text formatting consistently outperforms the MLP branch** — raw tokens give the model access to pre-trained world knowledge about specific political figures and parties; float true-rate aggregations discard this signal

2. **No freeze is better with text-format input** — the shifted input representation means epoch 1 gradients already point in a useful direction for the backbone; freezing wastes that capacity

3. **Epoch 3 is the sweet spot** — ep1 underfits, ep2 is close, ep3 peaks, ep4–5 overfit on 8k samples

4. **Class 0 recall improved significantly** — the model can exploit speaker identity tokens (e.g. known credible or unreliable sources from DeBERTa's pre-training), directly helping with the harder minority class

### What comes next

This architecture — text format, no freeze, LLRD, 3 epochs — became the **template for all k-fold experiments**:

| Notebook | Strategy | Holdout Macro F1 |
|---|---|---|
| **This notebook** | Single split, DeBERTa-v3-small | 0.6392 |
| `DeBERTa_KFold.ipynb` | 5-fold ensemble + late fusion → stacking | **0.6435** (fused) |
| `Mistral_LoRA_KFold.ipynb` | LoRA on Mistral-7B, 5-fold | **0.6553** (project best) |